# Modelling: property assessment valuation

This notebook trains four regression models to predict `log_value`
(log-transformed property assessment) for Calgary properties:

1. Ridge Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor
4. XGBoost Regressor

We compare MAE, RMSE, and R-squared, then tune XGBoost via cross-validation.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import cross_val_score, KFold

sys.path.insert(0, '..')
from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features
from src.model import (
    prepare_model_data,
    train_models,
    get_feature_importance,
    save_model,
)

DATA_DIR = str(Path('..') / 'data')
MODEL_DIR = str(Path('..') / 'models')
pd.set_option('display.max_columns', 40)
print('Setup complete.')

## 1. Prepare data

In [ ]:
raw = load_or_fetch_data(DATA_DIR, limit=100000)
df = preprocess_data(raw)
df = engineer_features(df)

X, y, label_encoders, feature_names = prepare_model_data(df)

print(f'Feature matrix: {X.shape}')
print(f'Target (log_value) mean: {y.mean():.4f}')
print(f'Features: {feature_names}')

## 2. Train/test split and train four regressors

The `train_models` function handles the 80/20 split, scales features for
Ridge, and computes original-scale MAE, RMSE, and log-scale R-squared.

In [ ]:
trained_models, results, scaler, X_test, y_test = train_models(X, y, random_state=42)

results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'MAE ($)', 'RMSE ($)', 'R2', 'MAPE (%)']
results_df = results_df.sort_values('R2', ascending=False)
results_df

In [ ]:
fig = px.bar(
    results_df, x='Model', y='R2', text='R2',
    title='R-squared comparison across models',
    color='Model', color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(height=400, showlegend=False, yaxis_range=[0, 1])
fig.show()

In [ ]:
fig = px.bar(
    results_df.melt(id_vars='Model', value_vars=['MAE ($)', 'RMSE ($)']),
    x='Model', y='value', color='variable', barmode='group',
    title='MAE and RMSE comparison (original dollar scale)',
    labels={'value': 'Dollars ($)', 'variable': 'Metric'},
)
fig.update_layout(height=400)
fig.show()

## 3. Cross-validate R-squared

Five-fold cross-validation on the full dataset to confirm the hold-out
R-squared estimates are stable.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=7, learning_rate=0.1, random_state=42, n_jobs=-1),
}

cv_results = []
for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='r2', n_jobs=-1)
    cv_results.append({'Model': name, 'Mean R2': round(scores.mean(), 4), 'Std': round(scores.std(), 4)})
    print(f'{name}: R2 = {scores.mean():.4f} (+/- {scores.std():.4f})')

cv_df = pd.DataFrame(cv_results)
cv_df

## 4. Tune XGBoost

A grid search over key hyperparameters for XGBoost.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [5, 7],
    'learning_rate': [0.05, 0.1],
}

xgb = XGBRegressor(random_state=42, n_jobs=-1)
grid = GridSearchCV(xgb, param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=0)
grid.fit(X, y)

print(f'Best params: {grid.best_params_}')
print(f'Best CV R2: {grid.best_score_:.4f}')

In [ ]:
grid_df = pd.DataFrame(grid.cv_results_)
grid_df = grid_df[['param_n_estimators', 'param_max_depth', 'param_learning_rate',
                    'mean_test_score', 'std_test_score']]
grid_df = grid_df.sort_values('mean_test_score', ascending=False)
grid_df.head(8)

## 5. Feature importance

In [ ]:
xgb_model = trained_models['XGBoost']
importance = get_feature_importance(xgb_model, feature_names, 'XGBoost')

fig = px.bar(importance, x='Importance', y='Feature', orientation='h',
             title='XGBoost feature importance',
             color='Importance', color_continuous_scale='YlOrRd')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
fig.show()

importance

## 6. Save best model

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = trained_models[best_name]
print(f'Best model: {best_name} (R2 = {results_df.iloc[0]["R2"]:.4f})')

save_model(best_model, scaler, label_encoders, feature_names, MODEL_DIR)
print(f'Model artifacts saved to {MODEL_DIR}')

## Summary

- All four regressors were trained on the same 80/20 split.
- XGBoost and Gradient Boosting achieve the highest R-squared (around 0.77).
- Cross-validation confirms the hold-out results are stable.
- Community-level aggregate features (`community_avg_value`, `community_median_value`) are the strongest predictors.

Detailed residual analysis and SHAP explanations follow in `04_evaluation.ipynb`.